# 데뷔 유형 분석 — 영어권 단독 vs 다중 대륙 동시 데뷔

**용도** — 첫 주 데뷔 형태(영어권 단독 19.4% vs 다중 대륙 동시 82.0%)가 글로벌 히트율을 갈라놓음을 집계하고, 단독 데뷔 작품의 데뷔 시장 분포를 확인한다.

**보고서 대응** — 보고서 핵심 요약(19.4%·82.0%) · 인사이트 1번(동시 공개 기본 설계)

**환경** — Python 3.15 / pandas 3.1.1 / scikit-learn 1.9.1 / LightGBM 4.7.0 / SHAP 0.54.1 (requirements.txt)

**실행 순서** — 00(개요) → 01(수집·결합) → 02(EDA) → 03(장르 보완) → 04(언어장벽) → 05(시장파워·미러) → 06(LightGBM) → 07(보정·SHAP) → 08(데뷔)

In [1]:
# 공통 설정 — 데이터는 원본 저장소에서 읽고, 출력물은 이 폴더에만 기록(원본 보호)
from pathlib import Path
import sys
ROOT = Path(r"C:\Users\USER\Desktop\엘리스 개인 프로젝트 자료\[엔터] Netflix Top 10 Weekly Dataset")
NBOUT = Path(r"C:\Users\USER\Desktop\엘리스 개인 프로젝트 자료\넷플릭스_분석_노트북\출력물")
NBOUT_F = NBOUT / "figures"
for _d in [NBOUT, NBOUT_F, NBOUT / "models", NBOUT / "data" / "processed", NBOUT / "data" / "interim"]:
    _d.mkdir(parents=True, exist_ok=True)
print("데이터 루트 =", ROOT)
print("출력 폴더  =", NBOUT)

데이터 루트 = C:\Users\USER\Desktop\엘리스 개인 프로젝트 자료\[엔터] Netflix Top 10 Weekly Dataset
출력 폴더  = C:\Users\USER\Desktop\엘리스 개인 프로젝트 자료\넷플릭스_분석_노트북\출력물


**원본 코드** — `데뷔 유형 분석 스크립트`

In [2]:
# -*- coding: utf-8 -*-
"""SHAP '데뷔영어권' 음수 신호 분해 (컬럼 순서 수정)"""
import pandas as pd
import numpy as np

D = r"C:\Users\USER\Desktop\엘리스 개인 프로젝트 자료\[엔터] Netflix Top 10 Weekly Dataset"
EN_MARKETS = {"United States", "United Kingdom", "Canada", "Australia", "Ireland", "New Zealand"}
old = pd.read_csv(D + r"\data\raw\all-weeks-countries.csv", encoding="cp1252", parse_dates=["week"])
new = pd.read_csv(D + r"\data\processed\all-weeks-countries_2022-2026.csv", parse_dates=["week"])
comb = pd.concat([old, new], ignore_index=True)
first = comb.groupby("show_title").week.min().rename("debut")
comb = comb.merge(first, on="show_title")
comb["offset_days"] = (comb.week - comb.debut).dt.days
comb["is_en_debut"] = comb.country_name.map(lambda c: c in EN_MARKETS)
debut_week = comb[comb.offset_days < 7]
g_set = set(pd.concat([pd.read_csv(D + r"\data\raw\all-weeks-global.csv", parse_dates=["week"]),
                       pd.read_csv(D + r"\data\processed\all-weeks-global_2022-2026.csv", parse_dates=["week"])],
                      ignore_index=True).show_title.unique())

deb = debut_week.groupby("show_title").agg(
    진입국가수=("country_name", "nunique"),
    영어권포함=("is_en_debut", "max"),
    비영어포함=("country_name", lambda s: int(any(c not in EN_MARKETS for c in s))))
deb["유형"] = np.where(deb.영어권포함 & deb.비영어포함, "다중(영어+비영어)",
              np.where(deb.영어권포함, "영어권 전용 데뷔", "비영어권 전용 데뷔"))
deb["글로벌히트"] = [1 if t in g_set else 0 for t in deb.index]
deb["카테고리"] = comb.groupby("show_title").category.agg(lambda s: s.mode()[0]).reindex(deb.index)

print("=== 데뷔 유형별 글로벌 히트율 (SHAP '데뷔영어권' 음수 신호 분해) ===")
t = deb.groupby("유형").agg(작품수=("글로벌히트", "count"), 글로벌히트율=("글로벌히트", "mean")).round(3)
print(t.to_string())
print()
print("=== 세부: 유형 x 넷플릭스 카테고리 ===")
t2 = deb.groupby(["유형", "카테고리"]).agg(작품수=("글로벌히트", "count"),
                                     글로벌히트율=("글로벌히트", "mean")).round(3)
print(t2.to_string())
deb.to_csv(r"C:\Users\USER\Desktop\엘리스 개인 프로젝트 자료\넷플릭스_분석_노트북\출력물" + r"\data\processed\debut_type_analysis.csv", index=False, encoding="utf-8-sig")
print("\n저장: data/processed/debut_type_analysis.csv")


=== 데뷔 유형별 글로벌 히트율 (SHAP '데뷔영어권' 음수 신호 분해) ===
             작품수  글로벌히트율
유형                      
다중(영어+비영어)  1492   0.820
비영어권 전용 데뷔  9615   0.216
영어권 전용 데뷔   1379   0.194

=== 세부: 유형 x 넷플릭스 카테고리 ===
                   작품수  글로벌히트율
유형         카테고리               
다중(영어+비영어) Films   786   0.795
           TV      706   0.847
비영어권 전용 데뷔 Films  7064   0.202
           TV     2551   0.255
영어권 전용 데뷔  Films  1018   0.196
           TV      361   0.186

저장: data/processed/debut_type_analysis.csv
